# 03 — Captions (brutes/nettoyées) et métadonnées visuelles

```text
Captions (brutes + nettoyées) et métadonnées visuelles (objets, attributs, relations, contexte).

ANTI-FUITE PAR CONSTRUCTION : les captions sont écrites dans des fichiers PHYSIQUEMENT séparés :
  tables/kb_captions.jsonl           -> seules captions chargeables par un retriever
  tables/eval_refs_val.jsonl, eval_refs_test.jsonl   -> références d'évaluation (jamais indexées)
Les métadonnées d'une image de val/test sont dérivées de l'IMAGE ou des annotations d'objets uniquement
(jamais de ses captions). Les attributs extraits des captions (`caption_attributes`) n'existent que pour la KB.
```

> Projet : étude comparative de techniques RAG pour le captioning d'images (MS COCO) — **groupe G1**. Exécuter les notebooks dans l'ordre : 00 → 01 → 01b → 02 → … → 09. Les helpers partagés sont dans `common.py`, `visual.py`, `rag_data.py`.

In [1]:
import os
# --- paramètre : "config.yaml" = vraies données COCO ; "config_test.yaml" = mini-COCO synthétique (test sans réseau)
CONFIG = os.environ.get("RAG_CONFIG", "config.yaml")
os.environ["RAG_CONFIG"] = CONFIG
print("config utilisée :", CONFIG)

config utilisée : config.yaml


## Imports et constantes

In [13]:
import re
from collections import Counter, defaultdict
import pandas as pd
from PIL import Image
from common import (clean_caption, complexity_label, context_label, get_logger, load_config, mkdirs, paths, read_json,
                    set_seed, tokenize, write_json, write_jsonl)
from visual import named_colors
log = get_logger("meta")
COLOR_WORDS = {"red", "blue", "green", "yellow", "orange", "purple", "pink", "brown", "black", "white", "gray", "grey",
               "silver", "gold", "golden", "beige", "tan"}
SIZE_WORDS = {"large", "big", "small", "little", "tiny", "huge", "tall", "short", "long", "giant"}
MATERIAL_WORDS = {"wooden", "metal", "plastic", "glass", "stone", "brick", "leather", "concrete"}
STATE_WORDS = {"open", "closed", "empty", "full", "wet", "dry", "snowy", "sunny", "cloudy", "dirty", "clean", "old", "new"}
SHAPE_WORDS = {"round", "circular", "square", "rectangular", "triangular", "oval", "curved", "flat", "pointed", "spherical", "cylindrical"}
ATTR_LEX = {**{w: "color" for w in COLOR_WORDS}, **{w: "size" for w in SIZE_WORDS},
            **{w: "material" for w in MATERIAL_WORDS}, **{w: "state" for w in STATE_WORDS},
            **{w: "shape" for w in SHAPE_WORDS}}

### `caption_attributes()`

In [3]:
def caption_attributes(tokens):
    out = []
    for i, t in enumerate(tokens):
        if t in ATTR_LEX and i + 1 < len(tokens):
            out.append({"attr": t, "type": ATTR_LEX[t], "head": tokens[i + 1]})
    return out

### `box_rel()`
Relation spatiale géométrique entre deux boîtes normalisées [x0,y0,x1,y1] (a par rapport à b).

In [4]:
def box_rel(a, b):
    """Relation spatiale géométrique entre deux boîtes normalisées [x0,y0,x1,y1] (a par rapport à b)."""
    ax0, ay0, ax1, ay1 = a; bx0, by0, bx1, by1 = b
    iw, ih = min(ax1, bx1) - max(ax0, bx0), min(ay1, by1) - max(ay0, by0)
    inter = iw * ih if iw > 0 and ih > 0 else 0.0
    aa, ba = (ax1 - ax0) * (ay1 - ay0), (bx1 - bx0) * (by1 - by0)
    if inter > 0:
        if inter / max(1e-9, min(aa, ba)) >= 0.8:
            return "inside" if aa <= ba else "contains"
        hov = iw / max(1e-9, ax1 - ax0)
        if ay1 <= by0 + 0.3 * (by1 - by0) and ay1 >= by0 and hov >= 0.5 and ay0 < by0:
            return "on_top_of"
        return "overlapping"
    dx, dy = (ax0 + ax1) / 2 - (bx0 + bx1) / 2, (ay0 + ay1) / 2 - (by0 + by1) / 2
    if abs(dx) >= abs(dy):
        return "left_of" if dx < 0 else "right_of"
    return "above" if dy < 0 else "below"

### `size_label()`

In [5]:
def size_label(area_frac):
    return "small" if area_frac < 0.02 else "medium" if area_frac < 0.15 else "large"

### `main()`

In [6]:
def main():
    cfg = load_config(); p = paths(cfg); mkdirs(p); set_seed(cfg["seed"])
    imgs = pd.read_csv(p["tables"] / "images.csv")
    imgs_ok = imgs[imgs.status == "ok"].set_index("image_id")
    n_cap = cfg["captions_per_image"]

    ann_by_src, caps_by_src, cats = {}, {}, {}
    for src in ["train2017", "val2017"]:
        inst = read_json(p["raw"] / "annotations" / f"instances_{src}.json")
        caps = read_json(p["raw"] / "annotations" / f"captions_{src}.json")
        for c in inst["categories"]:
            cats[c["id"]] = c
        d = defaultdict(list)
        for a in inst["annotations"]:
            if a.get("iscrowd", 0) == 0:
                d[a["image_id"]].append(a)
        ann_by_src[src] = d
        dc = defaultdict(list)
        for a in caps["annotations"]:
            dc[a["image_id"]].append(a)
        caps_by_src[src] = dc

    kb_caps, ev_caps = [], {"val": [], "test": []}
    meta, rel_counter = [], Counter()
    for split in ["kb", "val", "test"]:
        recs = read_json(p["splits"] / f"{split}.json")["images"]
        for r in recs:
            iid, src = r["image_id"], r["source"]
            if iid not in imgs_ok.index:
                continue
            row = imgs_ok.loc[iid]
            # ---- captions : on garde les n_cap premières (ordre de caption_id), valeurs brutes inchangées
            cl = sorted(caps_by_src[src][iid], key=lambda a: a["id"])[:n_cap]
            attrs_all = []
            for a in cl:
                raw = a["caption"]
                if split == "kb":
                    toks = tokenize(raw)
                    kb_caps.append({"caption_id": a["id"], "image_id": iid, "caption_raw": raw,
                                    "caption_clean": clean_caption(raw), "n_tokens": len(toks)})
                    attrs_all += caption_attributes(toks)
                else:
                    ev_caps[split].append({"caption_id": a["id"], "image_id": iid, "caption_raw": raw})
            # ---- objets (annotations COCO) + attributs pixel par objet
            W, H = r["width"], r["height"]
            objs = []
            anns = sorted(ann_by_src[src][iid], key=lambda a: -a["area"])
            rgb = None
            if not bool(row.exif_corrected):
                try:
                    rgb = Image.open(p["art"] / row.path_rgb).convert("RGB")
                except Exception:
                    rgb = None
            for k, a in enumerate(anns):
                x, y, w, h = a["bbox"]
                box = [max(0, x / W), max(0, y / H), min(1, (x + w) / W), min(1, (y + h) / H)]
                area_frac = max(0.0, (box[2] - box[0]) * (box[3] - box[1]))
                o = {"category_id": a["category_id"], "category": cats[a["category_id"]]["name"],
                     "supercategory": cats[a["category_id"]]["supercategory"], "bbox_norm": [round(v, 4) for v in box],
                     "area_frac": round(area_frac, 5), "size": size_label(area_frac), "source": "coco_gt"}
                if rgb is not None and k < 8 and area_frac >= 0.01:
                    iw, ih = rgb.size
                    crop = rgb.crop((int(box[0] * iw), int(box[1] * ih), max(int(box[0] * iw) + 2, int(box[2] * iw)),
                                     max(int(box[1] * ih) + 2, int(box[3] * ih)))).resize((32, 32))
                    import numpy as np
                    col = named_colors(np.asarray(crop, dtype=np.float32))
                    o["dominant_color"] = next(iter(col))
                objs.append(o)
            sc = sorted({o["supercategory"] for o in objs})
            rels = []
            top = objs[:8]
            for i in range(len(top)):
                for j in range(len(top)):
                    if i == j:
                        continue
                    rl = box_rel(top[i]["bbox_norm"], top[j]["bbox_norm"])
                    if rl in ("overlapping",) and i > j:
                        continue
                    rels.append({"subj": top[i]["category"], "rel": rl, "obj": top[j]["category"]})
                    rel_counter[rl] += 1
            cat_counts = Counter(o["category"] for o in objs)
            meta.append({
                "image_id": iid, "split": split, "n_objects": len(objs), "category_counts": dict(cat_counts),
                "n_categories": len(cat_counts), "supercategories": sc, "complexity": complexity_label(len(objs)),
                "context": context_label(sc),
                "day_night": "night_or_dark" if bool(row.is_dark) else "day_or_bright",
                "brightness": float(row.brightness), "colorfulness": float(row.colorfulness),
                "dominant_colors": str(row.dominant_colors).split("|"),
                "objects": objs, "relations": rels,
                "caption_attributes": attrs_all if split == "kb" else None,   # KB uniquement (dérivé du texte)
                "metadata_sources": {"objects": "coco_gt_annotations",
                                     "attributes_pixel": "named_colors_hsv", "relations": "bbox_geometry"},
            })
    write_jsonl(p["tables"] / "kb_captions.jsonl", kb_caps)
    for s in ("val", "test"):
        write_jsonl(p["tables"] / f"eval_refs_{s}.jsonl", ev_caps[s])
    write_jsonl(p["tables"] / "metadata.jsonl", meta)
    log.info("kb_captions=%d | val_refs=%d | test_refs=%d | metadata=%d", len(kb_caps), len(ev_caps["val"]),
             len(ev_caps["test"]), len(meta))
    write_json(p["audit"] / "metadata_info.json", {
        "object_source": "COCO instance annotations (GT). ATTENTION: pour les requêtes val/test, ces objets sont un "
                         "ORACLE ; utiliser 03b_detect_objects.py (DETR) pour les métadonnées 'réalistes'.",
        "attribute_lexicon_sizes": {k: sum(1 for v in ATTR_LEX.values() if v == k) for k in sorted(set(ATTR_LEX.values()))},
        "relations_geometry": ["inside", "contains", "on_top_of", "overlapping", "left_of", "right_of", "above", "below"],
        "relation_counts": dict(rel_counter),
        "caption_selection": f"{n_cap} premières captions par caption_id (COCO en contient parfois 6-7)",
    })

## Lancer l'étape

In [7]:
import sys
sys.argv = ["notebook"]          # pas d'arguments CLI dans un notebook
try:
    main()
except SystemExit as e:               # certains scripts terminent par sys.exit(code)
    print("terminé, code de sortie :", e.code)

19:20:31 | INFO | kb_captions=50000 | val_refs=500 | test_refs=500 | metadata=10200


## Résultats

In [8]:
import json, pandas as pd
from IPython.display import Image, Markdown, display
from common import load_config, paths
cfg = load_config(); P = paths(cfg)

### Captions KB (brut vs nettoyé)

In [9]:
from common import read_jsonl
kbc = pd.DataFrame(read_jsonl(P["tables"] / "kb_captions.jsonl")); display(kbc.head(8))
print("références val/test (fichiers séparés, jamais indexés) :")
display(pd.DataFrame(read_jsonl(P["tables"] / "eval_refs_test.jsonl")).head(5))

,caption_id,image_id,caption_raw,caption_clean,n_tokens
0,755795,78,A lighted owl candle sits next to a clock.,a lighted owl candle sits next to a clock,9
1,758024,78,White clock sitting next to a statue of an owl.,white clock sitting next to a statue of an owl,10
2,759054,78,a close up of a clock near an owl candle holder,a close up of a clock near an owl candle holder,11
3,761571,78,An antique looking clock beside an owl with a ...,an antique looking clock beside an owl with a ...,12
4,762369,78,An owl candle and a clock sitting in front of ...,an owl candle and a clock sitting in front of ...,12
5,775939,113,a man and woman cut into a big cake,a man and woman cut into a big cake,9
6,780412,113,A man and woman standing in front of a cake.,a man and woman standing in front of a cake,10
7,780745,113,A women who is cutting into a cake.,a women who is cutting into a cake,8


références val/test (fichiers séparés, jamais indexés) :


,caption_id,image_id,caption_raw
0,647108,2261,A man on a blue raft attempting to catch a rid...
1,649928,2261,The boy is looking back at a wave in the ocean.
2,652670,2261,A young man is body surfing and paddling in th...
3,655658,2261,A man in a black shirt plays on an ocean wave.
4,655676,2261,A boy riding a boogie board in the ocean waves.


### Métadonnées d'une image

In [10]:
meta = read_jsonl(P["tables"] / "metadata.jsonl")
m = next(x for x in meta if x["split"] == "test")
print({k: m[k] for k in ["image_id", "split", "n_objects", "category_counts", "complexity", "context", "day_night", "dominant_colors"]})
print("objets :", [(o["category"], o["size"], o.get("dominant_color")) for o in m["objects"][:6]])
print("relations :", m["relations"][:6]); print("attributs texte (None pour val/test) :", m["caption_attributes"])

{'image_id': 2261, 'split': 'test', 'n_objects': 2, 'category_counts': {'person': 1, 'surfboard': 1}, 'complexity': 'simple', 'context': 'mixed', 'day_night': 'day_or_bright', 'dominant_colors': ['gray', 'green', 'blue']}
objets : [('person', 'medium', 'gray'), ('surfboard', 'small', 'gray')]
relations : [{'subj': 'person', 'rel': 'overlapping', 'obj': 'surfboard'}]
attributs texte (None pour val/test) : None


### Info métadonnées

In [11]:
print(json.dumps(json.load(open(P["audit"] / "metadata_info.json")), indent=1, ensure_ascii=False))

{
 "object_source": "COCO instance annotations (GT). ATTENTION: pour les requÃªtes val/test, ces objets sont un ORACLE ; utiliser 03b_detect_objects.py (DETR) pour les mÃ©tadonnÃ©es 'rÃ©alistes'.",
 "attribute_lexicon_sizes": {
  "color": 17,
  "material": 8,
  "size": 10,
  "state": 13
 },
 "relations_geometry": [
  "inside",
  "contains",
  "on_top_of",
  "overlapping",
  "left_of",
  "right_of",
  "above",
  "below"
 ],
 "relation_counts": {
  "overlapping": 27040,
  "on_top_of": 5156,
  "above": 27269,
  "left_of": 74935,
  "inside": 12512,
  "right_of": 74935,
  "contains": 12512,
  "below": 27269
 },
 "caption_selection": "5 premiÃ¨res captions par caption_id (COCO en contient parfois 6-7)"
}
